In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

In [ ]:
matplotlib.rcParams['savefig.dpi'] = 400  # Set the DPI for saving figures
matplotlib.rcParams['axes.labelweight'] = 'bold'  # Bold x and y labels
matplotlib.rcParams['axes.labelsize'] = 25  # Maximize font size of x and y labels
# matplotlib.rcParams['axes.grid'] = True  # Enable grid view
matplotlib.rcParams['axes.edgecolor'] = 'black'  # Border color
matplotlib.rcParams['axes.linewidth'] = 2  # Border thickness
# tight layout
matplotlib.rcParams['figure.autolayout'] = True

In [ ]:
dataset = pd.read_csv("<input csv file>")

In [ ]:
# Remove rows where vpn starts with package
dataset = dataset[~dataset.apply(lambda row: str(row['vpn']).startswith(row['package']) if pd.notna(row['vpn']) else False, axis=1)]
dataset = dataset[~dataset.apply(lambda row: str(row['tor_bridge']).startswith(row['package']) if pd.notna(row['tor_bridge']) else False, axis=1)]


In [ ]:
# remove rows where package starts with org.torproject
dataset = dataset[~dataset['package'].str.startswith('org.torproject')]

In [ ]:
dataset.shape

In [ ]:
# for any row where lower of tor_bridge has shadowsocks in it and vpn is null, replace vpn with dataset["tor_bridge"]
dataset.loc[dataset['tor_bridge'].str.lower().str.contains('shadowsocks') & dataset['vpn'].isnull(), 'vpn'] = dataset['tor_bridge']
# make the tor bridge column with shadowsocks empty
dataset.loc[
    dataset['tor_bridge'].fillna('').str.lower().str.contains('shadowsocks'), 
    'tor_bridge'
] = ''


In [ ]:
# remove any row with duplicate hash
dataset = dataset.drop_duplicates(subset='hash', keep='first')

In [ ]:
dataset.loc[dataset["tor"] == "r/tOr", "tor"] = np.nan


In [ ]:
    # Count distinct packages in descending order
package_counts = dataset['package'].value_counts().reset_index()
package_counts.columns = ['package', 'count']

# Determine categories efficiently
category_mapping = dataset.groupby('package').agg({
    'tor': lambda x: 'tor' if x.notna().any() else None,
    'tor_bridge': lambda x: 'tor_bridge' if x.notna().any() else None,
    'proxy': lambda x: 'proxy' if x.notna().any() else None,
    'vpn': lambda x: 'vpn' if x.notna().any() else None,
    'i2p' : lambda x: 'i2p' if x.notna().any() else None,
}).fillna('')

# Combine the categories into a single string
category_mapping['category'] = category_mapping[['tor', 'tor_bridge', 'proxy', 'vpn', 'i2p']].apply(
    lambda x: ', '.join(filter(None, x)), axis=1
)

# Merge the categories with the package counts
package_counts = package_counts.merge(category_mapping[['category']], left_on='package', right_index=True)

print(package_counts)

In [ ]:
package_counts.to_csv("<input csv file>", index=False)

In [ ]:
package_counts = pd.read_csv("<input csv file>")

In [ ]:
plt.hist

In [ ]:
row = dataset[dataset["vpn"].notna()].loc[dataset[dataset["vpn"].notna()]['score'].idxmax()]
print(row["package"])
print(row["user"])
print(row["tor"])
print(row["path"])
print(row["score"])

In [ ]:
dataset.to_csv("<input csv file>", index=False)

In [ ]:
year_totals_malware = {
    2009: 0,
    2010: 0,
    2011: 32,
    2012: 9970,
    2013: 219123,
    2014: 541250,
    2015: 131981,
    2016: 606736,
    2017: 123704,
    2018: 350343,
    2019: 432544,
    2020: 269584,
    2021: 426224,
    2022: 424749,
    2023: 88738,
    2024: 25399,
    2025: 2455
}

In [ ]:
pip install -U kaleido

In [ ]:
pip install -U plotly

In [ ]:
import plotly.graph_objects as go

data_copy = dataset.copy()


desired_families = ['autoins', 'smsreg', 'hiddad', 'smssend', 'umpay', 'triada', 'ewind', 'kreditspy', 'dianjin', 'traca', 'smsspy', 'dataeye', 'cryxos']
# # Filter to only 'autoins' family
data_copy = data_copy[data_copy['family'].isin(desired_families)]


# dropping all singleton
# data_copy = data_copy[data_copy['family'] == 'SINGLETON']



# Replace NaN values with 0 and non-NaN values with 1 for vpn, tor, and tor_bridge
data_copy['vpn'] = data_copy['vpn'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['tor'] = data_copy['tor'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['tor_bridge'] = data_copy['tor_bridge'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['proxy'] = data_copy['proxy'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['i2p'] = data_copy['i2p'].apply(lambda x: 1 if not pd.isna(x) else 0)

# set data_copy to have only columns, package, year, tor, tor_bridge, vpn, proxy
# data_copy = data_copy[['package', 'year', 'tor', 'tor_bridge', 'vpn', 'proxy', 'i2p']]
data_copy = data_copy[['family', 'year', 'tor', 'tor_bridge', 'vpn', 'proxy', 'i2p']]
# data_copy = data_copy[['hash', 'year', 'tor', 'tor_bridge', 'vpn', 'proxy', 'i2p']]

# print(data_copy.iloc[0])

data_copy = data_copy[data_copy['year'] >= 2019]

def get_category(row):
    categories = []
    if row['tor'] == 1:
        categories.append('Tor')
    if row['tor_bridge'] == 1:
        categories.append('TorPT')
    if row['vpn'] == 1:
        categories.append('VPN')
    if row['proxy'] == 1:
        categories.append('Proxy')
    if row['i2p'] == 1:
        categories.append('I2P')
    
    if not categories:
        return "none"
    
    return ', '.join(categories)

data_copy['category'] = data_copy.apply(get_category, axis=1)

# Find unique years and categories
years = sorted(data_copy['year'].unique())
categories = data_copy['category'].unique()

year_category_map = {}

color_palette = [
    "rgba(255, 182, 193, 0.7)",  # Light Pink
    "rgba(173, 216, 230, 0.7)",  # Light Blue
    "rgba(144, 238, 144, 0.7)",  # Light Green
    "rgba(255, 218, 185, 0.7)",  # Peach Puff
    "rgba(216, 191, 216, 0.7)",  # Thistle
    "rgba(255, 228, 181, 0.7)",  # Moccasin
    "rgba(222, 184, 135, 0.7)",  # Burlywood
    "rgba(176, 224, 230, 0.7)",  # Powder Blue
    "rgba(255, 160, 122, 0.7)",  # Light Salmon
    "rgba(255, 250, 205, 0.7)",  # Lemon Chiffon
    "rgba(240, 128, 128, 0.7)",  # Light Coral
    "rgba(135, 206, 250, 0.7)",  # Light Sky Blue
    "rgba(152, 251, 152, 0.7)",  # Pale Green
    "rgba(221, 160, 221, 0.7)",  # Plum
    "rgba(46, 139, 87, 0.7)",    # Sea Green
    "rgba(255, 105, 180, 0.7)",  # Hot Pink
    "rgba(100, 149, 237, 0.7)",  # Cornflower Blue
    "rgba(60, 179, 113, 0.7)",   # Medium Sea Green
    "rgba(255, 222, 173, 0.7)",  # Navajo White
    "rgba(238, 130, 238, 0.7)",  # Violet
    "rgba(255, 69, 0, 0.7)",     # Red Orange
    "rgba(255, 192, 203, 0.7)",  # Pink
    "rgba(34, 139, 34, 0.7)",    # Forest Green
    "rgba(255, 239, 213, 0.7)",  # Papaya Whip
    "rgba(186, 85, 211, 0.7)",   # Medium Orchid
    "rgba(255, 140, 0, 0.7)",    # Dark Orange
    "rgba(70, 130, 180, 0.7)",   # Steel Blue
    "rgba(0, 191, 255, 0.7)",    # Deep Sky Blue
    "rgba(255, 248, 220, 0.7)",  # Cornsilk
    "rgba(218, 112, 214, 0.7)",  # Orchid
    "rgba(255, 165, 0, 0.7)"     # Orange
]
category_color = {category: color for category, color in zip(categories, color_palette)}

# Populate the dictionary with unique year-category combinations
for year in years:
    for category in categories:
        key = f"{year} - {category}"
        year_category_map[key] = len(year_category_map)

labels = list(year_category_map.keys())

node_colors = []
for label in labels:
    _, category = label.split(" - ")
    print(f"{category} - {category_color[category]}")
    node_colors.append(category_color[category])

# Prepare Sankey data
sources = []
targets = []
values = []
link_colors = []


# packages = data_copy["package"].unique()
families = data_copy["family"].unique()
# hashes = data_copy["hash"].unique()

start_year = data_copy['year'].min()
end_year = data_copy['year'].max()


from collections import Counter

seen_transitions_by_family = {}

for family in families:
    family_data = data_copy[data_copy['family'] == family]
    family_years = sorted(family_data['year'].unique())
    seen_transitions = set()

    for i in range(len(family_years) - 1):
        y1 = family_years[i]
        y2 = family_years[i + 1]

        data_y1 = family_data[family_data['year'] == y1]
        data_y2 = family_data[family_data['year'] == y2]

        for _, row1 in data_y1.iterrows():
            cat1 = row1['category']

            for _, row2 in data_y2.iterrows():
                cat2 = row2['category']

                # Handle in-between years
                for y_gap in range(y1, y2 - 1):
                    intermediate = (y_gap, cat1, y_gap + 1, cat1)
                    if intermediate not in seen_transitions:
                        src_label = f"{y_gap} - {cat1}"
                        tgt_label = f"{y_gap + 1} - {cat1}"

                        sources.append(year_category_map[src_label])
                        targets.append(year_category_map[tgt_label])
                        values.append(1)
                        link_colors.append(category_color[cat1])
                        seen_transitions.add(intermediate)

                final_transition = (y1, cat1, y2, cat2)
                if final_transition not in seen_transitions:
                    src_label = f"{y1} - {cat1}"
                    tgt_label = f"{y2} - {cat2}"

                    sources.append(year_category_map[src_label])
                    targets.append(year_category_map[tgt_label])
                    values.append(1)
                    link_colors.append(category_color[cat1])
                    seen_transitions.add(final_transition)

                    if cat1 != cat2:
                        print(f"Transition in {family}: {y1} {cat1} → {y2} {cat2}")

    seen_transitions_by_family[family] = seen_transitions

categories = set([label.split(" - ")[1] for label in labels])
categories = list(categories)

# Define the custom order
custom_order = ["Tor", "TorPT", "VPN", "Proxy", "I2P"]

# Helper function to get the custom order index
def get_custom_order_index(element):
    # Split by comma and find the index of each element in the custom order
    return [custom_order.index(e.strip()) for e in element.split(', ')]

# Sort by count of elements first (number of commas + 1), then by the custom order
sorted_categories = sorted(categories, key=lambda x: (len(x.split(', ')), get_custom_order_index(x)))

years = set([int(label.split(" - ")[0]) for label in labels])

bar_size_years = {}

for label in labels:
    year = int(label.split(" - ")[0])
    category = label.split(" - ")[1]
    if year not in bar_size_years:
        bar_size_years[year] = {}

    if category not in bar_size_years[year]:
        idx = labels.index(label)
        bar_size_years[year][category] = max(sources.count(idx), targets.count(idx))

for year in bar_size_years:
    total = sum(bar_size_years[year].values())
    if total != 0:
        for category in bar_size_years[year]:
            bar_size_years[year][category] = bar_size_years[year][category] / total
    else:
        for category in categories:
            if category in bar_size_years:
                bar_size_years[year].pop(category)

    diff = 0
    for category in sorted_categories:
        if category in bar_size_years[year]:
            temp = bar_size_years[year][category]
            bar_size_years[year][category] = diff
            diff += temp

    max_cat = max(bar_size_years[year].values())
    min_cat = min(bar_size_years[year].values())

    start = 0.02
    end = 0.98

    if max_cat != min_cat:
        for category in bar_size_years[year]:
            bar_size_years[year][category] = start + (bar_size_years[year][category] - min_cat) * (end - start) / (max_cat - min_cat)
    elif len(categories) >= 1:
        for category in bar_size_years[year]:
            bar_size_years[year][category] = start + (end - start) / len(categories) * sorted_categories.index(category)


y_positions = []
for label in labels:
    year = int(label.split(" - ")[0])
    category = label.split(" - ")[1]
    y_positions.append(bar_size_years[year][category])


x_positions = [0.01 + (int(label.split(" - ")[0]) - min(years)) * (0.98) / (max(years) - min(years)) for label in labels]

to_remove = set(list(range(len(labels))))
to_remove -= set(sources)
to_remove -= set(targets)

change_map = {}
diff = 0
for idx in range(len(labels)):
    if idx in to_remove:
        # change_map[idx] = idx - diff
        labels.pop(idx - diff)
        x_positions.pop(idx - diff)
        y_positions.pop(idx - diff)
        node_colors.pop(idx - diff)
        diff += 1
    else:
        change_map[idx] = idx - diff

for idx in range(len(sources)):
    sources[idx] = change_map[sources[idx]]
    targets[idx] = change_map[targets[idx]]

fig = go.Figure(data=[go.Sankey(
    arrangement="snap",
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        x = x_positions,
        y = y_positions,
        color=node_colors
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors
    )
)])

fig.update_layout(
    title_text="Sankey Diagram of VPN, TOR, TorPT, Proxy, I2P Usage by Year",
    font_size=18,
    height=1200,
    width=6000
    
)

fig.write_json("<path-to-directory>")
fig.write_image("<path-to-directory>")
fig.write_html("<path-to-directory>")

fig.show()

In [ ]:
labels = fig.data[0].node.label

categories = set([label.split(" - ")[1] for label in labels])
categories = list(categories)

# Define the custom order
custom_order = ["Tor", "TorPT", "VPN", "Proxy", "I2P"]

# Helper function to get the custom order index
def get_custom_order_index(element):
    # Split by comma and find the index of each element in the custom order
    return [custom_order.index(e.strip()) for e in element.split(', ')]

# Sort by count of elements first (number of commas + 1), then by the custom order
sorted_categories = sorted(categories, key=lambda x: (len(x.split(', ')), get_custom_order_index(x)))

years = set([int(label.split(" - ")[0]) for label in labels])

bar_size_years = {}

source_nodes = fig.data[0].link.source
target_nodes = fig.data[0].link.target

for label in labels:
    year = int(label.split(" - ")[0])
    category = label.split(" - ")[1]
    if year not in bar_size_years:
        bar_size_years[year] = {}

    if category not in bar_size_years[year]:
        idx = labels.index(label)
        bar_size_years[year][category] = max(source_nodes.count(idx), target_nodes.count(idx))

for year in bar_size_years:
    total = sum(bar_size_years[year].values())
    if total == 0:
        continue
    
    for category in bar_size_years[year]:
        bar_size_years[year][category] = bar_size_years[year][category] / total

    diff = 0
    for category in sorted_categories:
        if category in bar_size_years[year]:
            temp = bar_size_years[year][category]
            bar_size_years[year][category] = diff
            diff += temp

    max_cat = max(bar_size_years[year].values())
    min_cat = min(bar_size_years[year].values())

    start = 0.02
    end = 0.98

    if len(bar_size_years[year].keys()) > 1:
        for category in bar_size_years[year]:
            bar_size_years[year][category] = start + (bar_size_years[year][category] - min_cat) * (end - start) / (max_cat - min_cat)


y_positions = []
for label in labels:
    year = int(label.split(" - ")[0])
    category = label.split(" - ")[1]
    y_positions.append(bar_size_years[year][category])


x_positions = [0.01 + (int(label.split(" - ")[0]) - min(years)) * (0.98) / (max(years) - min(years)) for label in labels]
# y_positions = [0.01 + (sorted_categories.index(label.split(" - ")[1])) * (0.84) / (len(categories) - 1) for label in labels]

In [ ]:
fig.data[0].node.x = x_positions
fig.data[0].node.y = y_positions

fig.update_layout(
    width=6000,
    height=1200
)

fig.write_json("<path-to-directory>")
fig.write_image("<path-to-directory>")
fig.write_html("<path-to-directory>")

fig.show()

In [ ]:
source_copy = []
target_copy = []
value_copy = []
link_color_copy = []

In [ ]:
proxy_idx = set()
for idx in range(len(labels)):
    if labels[idx].split(" - ")[1] == "Proxy":
        proxy_idx.add(idx)

print(proxy_idx)

In [ ]:
remove_thres = 0

removed = {k: 0 for k in proxy_idx}
for idx in range(len(sources)):
    if sources[idx] in proxy_idx and targets[idx] in proxy_idx:
        removed[sources[idx]] += 1
        if removed[sources[idx]] == remove_thres:
            removed[sources[idx]] = 0
            source_copy.append(sources[idx])
            target_copy.append(targets[idx])
            value_copy.append(values[idx])
            link_color_copy.append(link_colors[idx])
        continue

    source_copy.append(sources[idx])
    target_copy.append(targets[idx])
    value_copy.append(values[idx])
    link_color_copy.append(link_colors[idx])

In [ ]:
source_copy.count(0)

In [ ]:
fig = go.Figure(data=[go.Sankey(
    arrangement="snap",
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        x = x_positions,
        y = y_positions,
        color=node_colors,
        align="right"
    ),
    link=dict(
        source=source_copy,
        target=target_copy,
        value=value_copy,
        color=link_color_copy
    )
)])

fig.update_layout(font_size=26, height=1000, width=4000, font_weight=1000)

In [ ]:
fig.data[0].node.x = x_positions
fig.data[0].node.y = y_positions

fig.update_layout(
    width=6000,
    height=1200
)

fig.write_json("<path-to-directory>")
fig.write_image("<path-to-directory>")
fig.write_html("<path-to-directory>")

fig.show()